1. Module install

In [0]:

# install world bank module

%pip install wbgapi

2. Fetch GDP data

In [0]:
import pandas as pd
import wbgapi as wb
from pyspark.sql import functions as F

# 1. Define country codes (USA, Euro Area 'EMU', China, Japan, India)
countries = ["USA", "EMU", "CHN", "JPN", "IND"]

# 2. Indicator for GDP growth (annual percentage)
indicator = "NY.GDP.MKTP.KD.ZG"

# 3. Fetch historical data from 2000 to 2019
raw_gdp = wb.data.DataFrame(indicator, countries, time=range(2000, 2020), skipBlanks=True)

# 4. Reset index so 'economy' and time periods are standard columns
gdp_df = raw_gdp.reset_index()

# 5. Convert Pandas DataFrame to Spark DataFrame
gdp_spark_df = spark.createDataFrame(gdp_df)

# 6. Transform from wide time columns to a clean long format using stack
# wbgapi labels time columns like 'YR2000', 'YR2001', etc.
year_cols = [c for c in gdp_spark_df.columns if c != 'economy']
stack_expr = f"stack({len(year_cols)}, " + ", ".join([f"'{col[2:]}', `{col}`" for col in year_cols]) + ") as (Year, GDP_Growth_Pct)"

long_gdp_df = (
    gdp_spark_df
    .select("economy", F.expr(stack_expr))
    .withColumn("Year", F.col("Year").cast("int"))
    .withColumn("GDP_Growth_Pct", F.col("GDP_Growth_Pct").cast("decimal(18,2)"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

# 7. Write to Bronze layer in long format (overwrite for initial clean slate)
(
    long_gdp_df.write.format("delta")
    .mode("overwrite")
    .saveAsTable("bronze.gdp_macro_raw")
)

print("Initial GDP macro data successfully ingested in clean long format into bronze.gdp_macro_raw!")